# 실습 3: AgentCore Gateway로 Agent에 도구를 안전하게 연결하기 

## 개요

이 실습에서는 Amazon Bedrock AgentCore Gateway를 사용하여 조직에서 사용할 수 있는 도구를 고객 지원 Agent와 통합하는 방법을 알아봅니다.

[Model Context Protocol(MCP)](https://modelcontextprotocol.io/docs/getting-started/intro)은 애플리케이션이 Large Language Models(LLM)에 도구와 컨텍스트를 제공하는 방식을 표준화하는 개방형 프로토콜입니다.

[Amazon Bedrock AgentCore Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html)를 사용하면 개발자는 몇 줄의 코드만으로 API, Lambda 함수, 기존 서비스를 MCP 호환 도구로 변환하고 Gateway 엔드포인트를 통해 Agent에서 사용할 수 있도록 제공할 수 있습니다.


**워크숍 진행 과정:**

- **실습 1(완료):** Agent 프로토타입 만들기 - 작동하는 고객 지원 Agent 구축
- **실습 2(완료):** Memory로 기능 강화 - 대화 컨텍스트 및 개인화 추가
- **실습 3(현재):** Gateway 및 Identity로 확장 - Agent 간에 도구를 안전하게 공유
- **실습 4:** 프로덕션에 배포 - AgentCore Runtime 및 Observability 사용
- **실습 5:** Agent 성능 평가 - 온라인 평가를 통해 품질 모니터링
- **실습 6:** 사용자 인터페이스 구축 - 고객용 애플리케이션 만들기


### AgentCore Gateway와 도구 공유가 중요한 이유

현재 상태(실습 1~2): 각 Agent가 자체 도구 사본을 가지고 있습니다. 이 방식은 확장하기 어려우며 다음과 같은 문제가 발생합니다.

- 서로 다른 Agent 간 코드 중복
- 일관되지 않은 도구 동작 및 유지 관리 부담
- 중앙 집중식 보안 또는 액세스 제어 부재
- 여러 사용 사례로 확장하기 어려움

이 실습을 마치면 다음과 같은 사용 사례에 활용할 수 있는 중앙 집중식 재사용 가능 도구를 갖추게 됩니다.

- 고객 지원 Agent(현재 사용 사례)
- 영업 Agent(동일한 제품 정보와 고객 데이터 필요)
- 재고 Agent(동일한 제품 정보와 보증 확인 필요)
- 반품 처리 Agent(반품 정책과 고객 프로필 필요)

그 밖의 다양한 사용 사례에도 활용할 수 있습니다. 

### AgentCore Identity로 안전한 인증 추가

AgentCore Gateway에서는 인바운드 및 아웃바운드 연결을 모두 안전하게 인증해야 합니다. [AgentCore Identity](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/identity.html)는 Okta, Entra, Amazon Cognito와 같은 표준 Identity Provider를 지원하며, AWS 서비스와 Slack, Zoom 같은 서드 파티 애플리케이션 전반에서 원활한 Agent Identity 및 액세스 관리를 제공합니다. 이 실습에서는 AgentCore Gateway와 AgentCore Identity를 통합하여 인바운드 및 아웃바운드 인증으로 안전하게 연결하는 방법을 알아봅니다. 

인바운드 인증에서 AgentCore Gateway는 호출 중 전달된 OAuth 토큰을 분석하여 Gateway 내 도구에 대한 액세스 허용 여부를 결정합니다. 도구에서 외부 리소스에 액세스해야 하는 경우 AgentCore Gateway는 API Key, IAM 또는 OAuth Token을 통한 아웃바운드 인증으로 외부 리소스에 대한 액세스 허용 여부를 결정할 수 있습니다.

인바운드 권한 부여 흐름에서는 Agent 또는 MCP 클라이언트가 사용자의 IdP에서 생성된 OAuth access token을 추가하여 AgentCore Gateway의 MCP 도구를 호출합니다. 그러면 AgentCore Gateway가 OAuth access token을 검증하고 인바운드 권한 부여를 수행합니다.

AgentCore Gateway에서 실행되는 도구가 외부 리소스에 액세스해야 하는 경우 OAuth는 Gateway 대상의 리소스 자격 증명 공급자를 사용하여 다운스트림 리소스의 자격 증명을 가져옵니다. AgentCore Gateway는 다운스트림 API에 액세스할 수 있도록 호출자에게 권한 부여 자격 증명을 전달합니다.


## 실습 3 아키텍처

<div style="text-align:left">
    <img src="images/architecture_lab3_gateway.png" width="75%"/>
</div>

*이제 웹 검색 도구가 AgentCore Gateway에 중앙화되고 안전한 Identity 기반 액세스 제어가 적용됩니다. 여러 Agent와 사용 사례에서 동일한 도구를 안전하게 공유할 수 있습니다. 또한 다른 애플리케이션용으로 구축한 `check_warranty()` 도구를 재사용하고, 다른 애플리케이션에서도 사용할 수 있도록 `web_search()` 도구를 추가합니다. `get_product_info()`, `get_return_policy()`, `get_technical_support`는 고객 지원 사용 사례에 특화되어 있으므로 로컬 도구로 유지합니다.*

### 주요 기능
- **AWS Lambda 함수의 원활한 통합:** 이 예제에서는 Amazon Bedrock AgentCore Gateway를 사용하여 Agent를 기존 AWS Lambda 함수와 통합하고 제품 보증을 확인하며 고객 프로필을 가져오는 방법을 보여 줍니다.
- **Inbound Auth로 Gateway 엔드포인트 보호:** 유효한 JWT 토큰을 제공하는 Agent만 엔드포인트에 연결하여 도구를 사용할 수 있습니다.
- **MCP 엔드포인트를 사용하도록 Agent 구성:** Agent가 유효한 JWT 토큰을 받아 AgentCore Gateway에서 제공하는 MCP 엔드포인트에 연결할 때 사용합니다.

## 사전 요구 사항

* Python 3.12+
* 구성된 AWS 자격 증명
* [Amazon Bedrock](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html)에서 활성화된 Amazon Nova 2 Lite
* 실습 2 '고객 지원 Agent에 Memory 추가' 완료
* AWS 워크숍 계정에는 다음 리소스가 미리 생성되어 있습니다.
    - AWS Lambda 함수 
    - AWS Lambda 실행 IAM Role
    - AgentCore Gateway IAM Role
    - AWS Lambda 함수에서 사용하는 DynamoDB 테이블 
    - Cognito User Pool 및 User Pool Client

## 단계 1: 필수 라이브러리 가져오기

In [ ]:
# 라이브러리 가져오기
import os
import sys
import boto3
import json
import time

from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client
from lab_helpers.utils import (
    get_or_create_cognito_pool,
    put_ssm_parameter,
    get_ssm_parameter,
    load_api_spec,
)


sts_client = boto3.client("sts")
account_id = sts_client.get_caller_identity()["Account"]
# AWS 계정 세부 정보 가져오기
REGION = boto3.session.Session().region_name

gateway_client = boto3.client(
    "bedrock-agentcore-control",
    region_name=REGION,
)

print("✅ Libraries imported successfully!")

## 단계 2: 기존 고객 데이터에 액세스할 수 있는 도구를 Agent에 제공
AgentCore Gateway는 다음 세 가지 주요 방식으로 Agent 도구 통합을 간소화합니다.

범용 MCP 지원: AgentCore Gateway의 MCP 표준을 통해 도구를 노출하여 모든 Agent 프레임워크와 즉시 호환

간편한 REST 통합: 기존 REST 서비스를 AgentCore Gateway 대상으로 추가하기만 하면 Agent 도구로 변환

Lambda 유연성: 모든 API를 호출할 수 있는 Lambda 함수를 MCP 엔드포인트로 노출 - 여기서는 보증 상태를 확인하는 함수로 시연

AgentCore Gateway는 호출할 도구 이름을 Lambda context에 입력하고 도구에 전달할 파라미터를 Lambda event에 제공합니다.

```
extended_tool_name = context.client_context.custom["bedrockAgentCoreToolName"]
resource = extended_tool_name.split("___")[1]
```

[Lambda 함수](./prerequisite/lambda/python/lambda_function.py)

```
def lambda_handler(event, context):
    if get_tool_name(event) == "check_warranty_status":
        serial_number = get_named_parameter(event=event, name="serial_number")
        customer_email = get_named_parameter(event=event, name="customer_email")

        warranty_status = check_warranty_status(serial_number, customer_email)
        return {"statusCode": 200, "body": warranty_status}
```

## 단계 3: 웹 검색 도구를 MCP로 변환
AgentCore Gateway를 사용하여 MCP 서버를 개발하고 있으므로 여러 Agent에서 사용할 도구를 MCP 도구로 변환할 수 있습니다. 실습 1에서 구축한 웹 검색 도구가 그 예입니다. 이에 따라 실습 1의 웹 검색 도구를 AgentCore Gateway 내의 Lambda 도구로 변환했습니다.

[웹 검색 Lambda](./prerequisite/lambda/python/web_search.py)
```
from ddgs import DDGS


def web_search(keywords: str, region: str = "us-en", max_results: int = 5) -> str:
    """Search the web for updated information.
    
    Args:
        keywords (str): The search query keywords.
        region (str): The search region: wt-wt, us-en, uk-en, ru-ru, etc.
        max_results (int): The maximum number of results to return.
        
    Returns:
        List of dictionaries with search results.
    """
    try:
        results = DDGS().text(keywords, region=region, max_results=max_results)
        return results if results else "No results found."
    except Exception as e:
        return f"Search error: {str(e)}"


print("✅ Web search tool ready")
```

## 단계 4: 함수 정의 메타데이터 생성
마지막으로 Lambda 함수에서 구현한 도구를 설명하는 도구 스키마를 작성해야 합니다.

이 파일은 [prerequisite/lambda/api_spec.json](./prerequisite/lambda/api_spec.json)에 이미 정의되어 있습니다.

```
[
    {
        "name": "check_warranty_status",
        "description": "Check the warranty status of a product using its serial number and optionally verify via email",
        "inputSchema": {
            "type": "object",
            "properties": {
                "serial_number": {
                    "type": "string"
                },
                "customer_email": {
                    "type": "string"
                }
            },
            "required": [
                "serial_number"
            ]
        }
    },
    {
        "name": "web_search",
        "description": "Search the web for updated information using DuckDuckGo",
        "inputSchema": {
            "type": "object",
            "properties": {
                "keywords": {
                    "type": "string",
                    "description": "The search query keywords"
                },
                "region": {
                    "type": "string",
                    "description": "The search region (e.g., us-en, uk-en, ru-ru)"
                },
                "max_results": {
                    "type": "integer",
                    "description": "The maximum number of results to return"
                }
            },
            "required": [
                "keywords"
            ]
        }
    }
]
```

## 단계 5: AgentCore Gateway 생성

이제 Lambda 함수를 MCP 호환 엔드포인트로 노출할 AgentCore Gateway를 생성합니다.

도구를 호출할 권한이 있는 호출자를 검증하려면 Inbound Auth를 구성해야 합니다.

Inbound Auth는 MCP 서버의 표준인 OAuth 권한 부여를 사용합니다. OAuth를 사용하는 경우 클라이언트 애플리케이션은 Gateway를 사용하기 전에 OAuth authorizer를 통해 인증해야 합니다. 클라이언트는 런타임에 사용할 access token을 받습니다.

OAuth discovery 서버와 client ID를 지정해야 합니다. 워크숍에서 제공하는 CloudFormation은 Cognito UserPool 및 UserPoolClient를 이미 프로비저닝했으며 discovery URL과 Client ID를 전용 SSM 파라미터에 저장했습니다.

In [ ]:
gateway_name = "customersupport-gw"

cognito_config = get_or_create_cognito_pool(refresh_token=True)
auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [cognito_config["client_id"]],
        "discoveryUrl": cognito_config["discovery_url"],
    }
}

try:
    # 새 Gateway 생성
    print(f"Creating gateway in region {REGION} with name: {gateway_name}")

    create_response = gateway_client.create_gateway(
        name=gateway_name,
        roleArn=get_ssm_parameter("/app/customersupport/agentcore/gateway_iam_role"),
        protocolType="MCP",
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration=auth_config,
        description="Customer Support AgentCore Gateway",
    )

    gateway_id = create_response["gatewayId"]

    gateway = {
        "id": gateway_id,
        "name": gateway_name,
        "gateway_url": create_response["gatewayUrl"],
        "gateway_arn": create_response["gatewayArn"],
    }
    put_ssm_parameter("/app/customersupport/agentcore/gateway_id", gateway_id)
    put_ssm_parameter("/app/customersupport/agentcore/gateway_name", gateway_name)
    put_ssm_parameter("/app/customersupport/agentcore/gateway_arn", create_response["gatewayArn"])
    put_ssm_parameter("/app/customersupport/agentcore/gateway_url", create_response["gatewayUrl"])

    time.sleep(3)

    print(f"✅ Gateway created successfully with ID: {gateway_id}")

except Exception:
    # Gateway가 있으면 SSM에서 기존 Gateway ID 가져오기
    existing_gateway_id = get_ssm_parameter("/app/customersupport/agentcore/gateway_id")
    print(f"Found existing gateway with ID: {existing_gateway_id}")

    # 기존 Gateway 세부 정보 가져오기
    gateway_response = gateway_client.get_gateway(gatewayIdentifier=existing_gateway_id)
    gateway = {
        "id": existing_gateway_id,
        "name": gateway_response["name"],
        "gateway_url": gateway_response["gatewayUrl"],
        "gateway_arn": gateway_response["gatewayArn"],
    }
    gateway_id = gateway["id"]

## 단계 6: Lambda 함수 대상 추가
이제 [prerequisite/lambda/api_spec.json](./prerequisite/lambda/api_spec.json)에 앞서 정의한 함수 정의를 사용하여 Agent Gateway 내에 Lambda 대상을 생성합니다. 이를 통해 Gateway에서 호스팅할 도구를 정의합니다.

Gateway에는 여러 대상을 연결할 수 있으며 연결된 대상과 도구를 언제든지 변경할 수 있습니다. 각 대상은 자체 자격 증명 공급자를 사용할 수 있지만, Gateway는 수많은 API에 걸쳐 Agent와 관련된 모든 도구에 액세스할 수 있는 단일 MCP URL을 제공합니다.

In [ ]:
try:
    api_spec_file = "./prerequisite/lambda/api_spec.json"

    # API 사양 파일이 있는지 확인
    if not os.path.exists(api_spec_file):
        print(f"❌ API specification file not found: {api_spec_file}")
        sys.exit(1)

    api_spec = load_api_spec(api_spec_file)

    # Gateway의 Inbound OAuth에 Cognito 사용
    lambda_target_config = {
        "mcp": {
            "lambda": {
                "lambdaArn": get_ssm_parameter("/app/customersupport/agentcore/lambda_arn"),
                "toolSchema": {"inlinePayload": api_spec},
            }
        }
    }

    # Gateway 대상 생성
    credential_config = [{"credentialProviderType": "GATEWAY_IAM_ROLE"}]

    create_target_response = gateway_client.create_gateway_target(
        gatewayIdentifier=gateway_id,
        name="LambdaUsingSDK",
        description="Lambda Target using SDK",
        targetConfiguration=lambda_target_config,
        credentialProviderConfigurations=credential_config,
    )

    print(f"✅ Gateway target created: {create_target_response['targetId']}")

except Exception as e:
    print(f"❌ Error creating gateway target: {str(e)}")

## 단계 7: 새로운 MCP 기반 도구를 지원 Agent에 추가
여기서는 Cognito의 인증 토큰을 Strands SDK의 MCPClient에 통합하여 Strands Agent와 연동할 MCP Server 객체를 생성합니다.
### 단계 7.1: 안전한 MCP 클라이언트 객체 설정

In [ ]:
print(f"Gateway Endpoint - MCP URL: {gateway['gateway_url']}")
# MCP 클라이언트 설정
mcp_client = MCPClient(
    lambda: streamablehttp_client(
        gateway["gateway_url"],
        headers={"Authorization": f"Bearer {cognito_config['bearer_token']}"},
    )
)

with mcp_client:
    tools = mcp_client.list_tools_sync()
    print(f"   Found {len(tools)} tool(s):\n")
    for tool in tools:
        print(f"   ✅ {tool.mcp_tool.name}")
        print(f"      {tool.mcp_tool.description}\n")

## 단계 7.2: MCP 클라이언트를 사용하여 Agent에서 도구에 액세스
이제 이전 실습의 리소스와 앞에서 구축한 AgentCore Gateway를 사용하여 Strands Agent를 생성합니다. 이제 Agent는 Strands Agent를 통한 로컬 도구와 AgentCore Gateway를 통한 MCP 도구를 함께 사용합니다.

In [ ]:
from lab_helpers.lab1_strands_agent import (
    get_product_info,
    get_return_policy,
    get_technical_support,
    SYSTEM_PROMPT,
)

import uuid
from lab_helpers.lab2_memory import create_or_get_memory_resource
from bedrock_agentcore.memory.integrations.strands.config import (
    AgentCoreMemoryConfig,
    RetrievalConfig,
)
from bedrock_agentcore.memory.integrations.strands.session_manager import (
    AgentCoreMemorySessionManager,
)

memory_id = create_or_get_memory_resource()

SESSION_ID = str(uuid.uuid4())
CUSTOMER_ID = "customer_001"

memory_config = AgentCoreMemoryConfig(
    memory_id=memory_id,
    session_id=str(SESSION_ID),
    actor_id=CUSTOMER_ID,
    retrieval_config={
        "support/customer/{actorId}/semantic/": RetrievalConfig(top_k=3, relevance_score=0.2),
        "support/customer/{actorId}/preferences/": RetrievalConfig(top_k=3, relevance_score=0.2),
    },
)

# Bedrock 모델 초기화
model_id = "global.amazon.nova-2-lite-v1:0"
model = BedrockModel(
    model_id=model_id,
    temperature=0.3,  # 창의성과 일관성의 균형
    region_name=REGION,
)


def create_agent(prompt):
    try:
        with mcp_client:
            tools = [
                get_product_info,
                get_return_policy,
                get_technical_support,
            ] + mcp_client.list_tools_sync()

            # 고객 지원 Agent 생성
            agent = Agent(
                model=model,
                tools=tools,
                system_prompt=SYSTEM_PROMPT,
                session_manager=AgentCoreMemorySessionManager(memory_config, REGION),
            )
            response = agent(prompt)
            return response
    except Exception as e:
        raise e


print("✅ Customer support agent created successfully!")

## 단계 8: 기존 API에 액세스하는 MCP 도구로 Agent 테스트

샘플 질의로 Agent를 테스트하여 모든 기능이 올바르게 작동하는지 확인합니다. 출력에 다섯 가지 도구가 표시되어야 합니다.

In [ ]:
test_prompts = [
    # 보증 확인
    "List all of your tools",
    "I bought an iphone 14 last month. I don't like it because it heats up. How do I solve it?",
    "I have a Gaming Console Pro device , I want to check my warranty status, warranty serial number is MNO33333333.",
    "What are the warranty support guidelines?",
    "How can I fix Lenovo Thinkpad with a blue screen",
    "Tell me detailed information about the technical documentation on installing a new CPU",
]


# Agent 테스트 함수
def test_agent_responses(prompts):
    for i, prompt in enumerate(prompts, 1):
        print(f"\nTest Case {i}: {prompt}")
        print("-" * 50)
        try:
            response = create_agent(prompt)
            print(response)
        except Exception as e:
            print(f"Error: {str(e)}")
        print("-" * 50)


# 테스트 실행
test_agent_responses(test_prompts)

print("\\n✅ Basic testing completed!")

## [선택 사항] AgentCore Policy

### 단계 9: Policy Engine 생성

세분화된 액세스 제어를 위한 Cedar 권한 부여 정책을 포함할 Policy Engine을 생성합니다.

In [ ]:
# Toolkit에서 가져오고, 사용할 수 없으면 사용자 지정 구현 사용
try:
    from bedrock_agentcore_starter_toolkit.operations.policy.client import PolicyClient

    print("✅ Using toolkit PolicyClient")
except ImportError:
    from utils.policy_utils import PolicyClient

    print("✅ Using custom PolicyClient (toolkit policy module not available)")

# Policy 클라이언트 초기화
policy_client = PolicyClient(region_name=REGION)

print("\n🔧 Creating Policy Engine...")

# Policy Engine을 생성하거나 기존 항목 가져오기
# Policy Engine은 모든 권한 부여 정책을 담는 컨테이너입니다.
engine = policy_client.create_or_get_policy_engine(
    name="customersupport_pe",
    description="Policy engine for customer support gateway",
)

engine_id = engine["policyEngineId"]
engine_arn = engine["policyEngineArn"]
put_ssm_parameter("/app/customersupport/agentcore/policy_engine_id", engine_id)

print("\n✅ Policy Engine ready")
print(f"   Engine ID: {engine_id}")
print(f"   Engine ARN: {engine_arn}")

### [선택 사항]: 자연어로 Cedar Policy 생성

AgentCore Policy는 자연어를 사용하여 Cedar Policy를 생성하고 범위 기반 액세스를 적용하는 기능을 제공합니다.

In [ ]:
import time

# approve 도구용 Cedar Policy 생성(write 범위)
print("\n📝 Generating Cedar Policy from Natural language...")

nl_input = "Allow tag username == 'testuser' to perform check warranty status on the customer support gateway."

warranty_tool_policy = policy_client.generate_policy(
    policy_engine_id=engine["policyEngineId"],
    name=f"nl_policy_{int(time.time())}",
    resource={"arn": gateway["gateway_arn"]},
    content={"rawText": nl_input},
    fetch_assets=True,
)

print("✅ Policy generated from natural language")

In [ ]:
print("📋 Generated Cedar Policies:\n")
print("=" * 80)

# 보증 상태 허용 정책
print("\n1️⃣  Warranty Status")
print("-" * 80)
warranty_tool_policy_cedar = warranty_tool_policy["generatedPolicies"][0]["definition"]["cedar"]["statement"]
print(warranty_tool_policy_cedar)

print("\n" + "=" * 80)

### 단계 10: Cedar Policy 생성

범위 기반 액세스를 적용할 Cedar Policy를 생성합니다. approve 작업에는 write 범위를, create/list 작업에는 read 범위를 사용할 수 있습니다.

이 워크숍에서는 모든 사용자가 일관된 결과를 얻도록 미리 작성된 허용/거부 Cedar Policy를 사용합니다. 

In [ ]:
# 사용 사례 실행의 일관성을 위한 정책 제공
allow_policy = {
    "cedar": {
        "statement": f"""permit(
            principal,
            action in [AgentCore::Action::"LambdaUsingSDK___check_warranty_status", AgentCore::Action::"LambdaUsingSDK___web_search"],
            resource == AgentCore::Gateway::"arn:aws:bedrock-agentcore:{REGION}:{account_id}:gateway/{gateway["id"]}"
        ) when {{
            (principal.hasTag("username")) && 
            ((principal.getTag("username")) == "testuser")
        }};"""
    }
}

# "iPhone 8" 키워드에 대한 웹 검색 거부
deny_web_search_policy = {
    "cedar": {
        "statement": f"""forbid(
            principal,
            action == AgentCore::Action::"LambdaUsingSDK___web_search",
            resource == AgentCore::Gateway::"arn:aws:bedrock-agentcore:{REGION}:{account_id}:gateway/{gateway["id"]}"
        ) when {{
            context.input has keywords &&
            context.input.keywords like "*iPhone 8*"
        }};"""
    }
}

### 단계 11: Policy Engine에 정책 추가

생성된 Cedar Policy를 Policy Engine에 추가합니다.

In [ ]:
print("🔧 Creating policies in Policy Engine...\n")

import time as _time


def _cleanup_failed_policy(pe_id, name):
    """깔끔하게 재시도할 수 있도록 지정한 이름의 CREATE_FAILED 정책을 삭제합니다."""
    try:
        cp_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
        existing = cp_client.list_policies(policyEngineId=pe_id)
        for p in existing.get("policies", []):
            if p["name"] == name and p["status"] == "CREATE_FAILED":
                cp_client.delete_policy(policyEngineId=pe_id, policyId=p["policyId"])
                print(f"   Deleted stale CREATE_FAILED policy: {name}")
                _time.sleep(2)
    except Exception:
        pass


# 두 도구를 모두 허용하는 정책 생성
try:
    warranty_result = policy_client.create_or_get_policy(
        policy_engine_id=engine["policyEngineId"],
        name="allow_policy",
        description="Allow web_search and check_warranty_status calls",
        definition=allow_policy,
    )
    print("✅ Policy ready: allow_policy")
except Exception as e:
    print(f"⚠️  Policy creation failed: {e}")
    print("   Retrying with validation_mode='IGNORE_ALL_FINDINGS'...")
    _cleanup_failed_policy(engine["policyEngineId"], "allow_policy")
    warranty_result = policy_client.create_or_get_policy(
        policy_engine_id=engine["policyEngineId"],
        name="allow_policy",
        description="Allow web_search and check_warranty_status calls",
        definition=allow_policy,
        validation_mode="IGNORE_ALL_FINDINGS",
    )
    print("✅ Policy ready with IGNORE_ALL_FINDINGS: allow_policy")
print("   Tools allowed: check_warranty_status and web_search\n")

# 웹 검색 목록을 거부하는 정책 생성
try:
    web_search_deny_result = policy_client.create_or_get_policy(
        policy_engine_id=engine["policyEngineId"],
        name="deny_web_search",
        description="Deny web_search tool call for iPhone 8",
        definition=deny_web_search_policy,
    )
    print("✅ Policy ready: deny_web_search")
except Exception as e:
    print(f"⚠️  Policy creation failed: {e}")
    print("   Retrying with validation_mode='IGNORE_ALL_FINDINGS'...")
    _cleanup_failed_policy(engine["policyEngineId"], "deny_web_search")
    web_search_deny_result = policy_client.create_or_get_policy(
        policy_engine_id=engine["policyEngineId"],
        name="deny_web_search",
        description="Deny web_search tool call for iPhone 8",
        definition=deny_web_search_policy,
        validation_mode="IGNORE_ALL_FINDINGS",
    )
    print("✅ Policy ready with IGNORE_ALL_FINDINGS: deny_web_search")
print("   Tools denied conditionally: web_search\n")

print("✅ All policies ready!")

### 단계 12: Gateway IAM Role 권한 업데이트

Policy Engine을 연결하기 전에 Gateway IAM Role에 Policy Engine 액세스 권한을 부여해야 합니다.

In [ ]:
role_arn = get_ssm_parameter("/app/customersupport/agentcore/gateway_iam_role")
role_name = role_arn.split("/")[-1]

iam_client = boto3.client("iam")
print("🔧 Updating Gateway IAM role with Policy Engine permissions...")

# Policy Engine 액세스 권한을 부여하는 정책 문서
policy_document = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["bedrock-agentcore:*"],
            "Resource": [
                f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:policy-engine/*",
                f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:gateway/*",
            ],
        }
    ],
}

try:
    # Role에 인라인 정책 추가
    iam_client.put_role_policy(
        RoleName=role_name,
        PolicyName="PolicyEngineAccess",
        PolicyDocument=json.dumps(policy_document),
    )

    print("✅ IAM role updated successfully")
    print(f"   Role: {role_name}")
    print("   Added permissions: GetPolicyEngine, GetPolicy, ListPolicies")
    print("\n⏳ Waiting 10 seconds for IAM changes to propagate...")
    time.sleep(10)

    print("✅ Ready to attach Policy Engine")

except Exception as e:
    print(f"❌ Error updating IAM role: {e}")
    print("\nYou may need to manually add these permissions to the role.")

### 단계 13: Gateway에 Policy Engine 연결

Policy Engine을 ENFORCE 모드로 Gateway에 연결하여 정책 적용을 활성화합니다.

In [ ]:
from bedrock_agentcore_starter_toolkit.operations.gateway.client import GatewayClient

# Gateway 클라이언트 초기화
gateway_client_toolkit = GatewayClient(region_name=REGION)

print("🔧 Attaching Policy Engine to Gateway...")
print("   Mode: ENFORCE (policies will block unauthorized requests)\n")

# Policy Engine을 Gateway에 연결
update_response = gateway_client_toolkit.update_gateway_policy_engine(
    gateway_identifier=gateway["id"],
    policy_engine_arn=engine["policyEngineArn"],
    mode="ENFORCE",
)

print("✅ Policy Engine attached successfully!")
print(f"   Gateway ID: {gateway['id']}")
print(f"   Policy Engine: {engine['policyEngineId']}")
print("   Mode: ENFORCE")
print("\n🔒 Authorization is now active!")

### 단계 14: 정책 적용 테스트

In [ ]:
test_prompts = [
    "List all of your tools",
    "Search the web for heating issues with Samsung zfold 7",
    "Search the internet for heating issues with iPhone 8",
]


# Agent 테스트 함수
def test_agent_responses(prompts):
    for i, prompt in enumerate(prompts, 1):
        print(f"\nTest Case {i}: {prompt}")
        print("-" * 50)
        try:
            response = create_agent(prompt)
            print(response)
        except Exception as e:
            print(f"Error: {str(e)}")
        print("-" * 50)


# 테스트 실행
test_agent_responses(test_prompts)

print("\\n✅ Policy testing completed!")

### 축하합니다! 🎉


실습 3 'AgentCore Gateway로 Agent에 도구를 안전하게 연결하기'를 성공적으로 완료했습니다.

완료한 작업:

##### 도구 중앙화 및 재사용성

- 웹 검색을 로컬 도구에서 중앙 집중식 AgentCore Gateway로 마이그레이션
- 기존 엔터프라이즈 Lambda 함수(보증 확인, 고객 프로필) 통합
- 여러 유형의 Agent가 액세스할 수 있는 공유 도구 인프라 생성

##### 엔터프라이즈급 보안

- Cognito 통합을 통한 JWT 기반 인증 구현
- Gateway 액세스를 위한 안전한 인바운드 권한 부여 구성
- 도구 사용을 위한 Identity 기반 액세스 제어 설정

##### 확장 가능한 아키텍처 기반

- 여러 사용 사례(고객 지원, 영업, 반품 처리)를 지원하는 재사용 가능 도구 구축
- 서로 다른 Agent 간 코드 중복 제거
- 도구 업데이트 및 유지 관리를 위한 중앙 집중식 관리 체계 구축

##### 현재 제한 사항(다음 실습에서 해결합니다!)

- **로컬 개발 환경** - 여전히 노트북에서 실행되므로 프로덕션 준비가 되지 않았습니다.
- **제한된 Observability** - Agent 동작과 성능을 종합적으로 모니터링할 수 없습니다.
- **수동 확장** - 증가한 부하나 여러 동시 사용자를 자동으로 처리할 수 없습니다.

##### 다음 실습: 실습 4 - AgentCore Runtime으로 프로덕션에 배포

실습 4에서는 다음 기능을 추가하여 프로토타입을 프로덕션 지원 시스템으로 전환합니다.

- 확장 가능한 Agent 배포를 위한 AgentCore Runtime
- 지표, 로깅, 추적을 통한 종합적인 Observability
- 실제 트래픽을 처리하는 자동 확장 기능

### 리소스
- [Amazon Bedrock Agent Core Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html)
- [Strands Agents 문서](https://github.com/strands-agents/sdk-python)
- [공식 고객 지원 샘플](https://github.com/awslabs/amazon-bedrock-agentcore-samples/tree/main/02-use-cases/customer-support-assistant)